> # ⚠️ Do not run this notebook unattended
>
> The last section, **`# Hardware validation`**, connects to the real board, rewrites the
> TMP102's registers, and has steps where you must act (pinch the sensor) or keep your hands
> off it. *Run All* and `jlx run` reach that section.
>
> - **Safe to run on its own:** everything above `# Hardware validation`. It runs on the host
>   against a fake part, writes the module and its tests, and needs no board.
> - **Hardware section:** run it one cell at a time, reading the **What to do** line above
>   each cell first.

**Runs on the host first, the board last.** Everything down to `# Hardware validation` runs
on CPython (the notebook is not attached to a board): it writes the module and its tests and
checks them against a fake part. The last section connects to the FeatherS3, uploads the
freshly written package, checks the real TMP102 behind the mux with a matching test for every
unit test above, then disconnects.

micropylab has `%%extract` built in, so there is no `%load_ext jupyter_lab_extractor` cell.

# TODO:
Nothing

Decided (2026-11-13): this notebook replaces `notebooks/01_tmp102_bringup` (its hardware
checks are the `# Hardware validation` section). Unit tests run on the host against
`micropython_i2c.fakes.FakeTMP102`; hardware checks run on the board.

# Python file
location: `src/micropython_i2c/drivers/tmp102.py`

tests: `tests/test_tmp102.py`

## header

In [1]:
%%extract ../src/micropython_i2c/drivers/tmp102.py
"""TI TMP102 digital temperature sensor (SBOS397I).

Usage on the board::

    from micropython_i2c import I2CConnection
    from micropython_i2c.drivers.tmp102 import TMP102

    tmp = TMP102(I2CConnection(bus, 0x48, lock=bus_lock))
    print(tmp.temperature())        # -> 23.4375

Two classes:

* ``TMP102Regs``: datasheet facts only (addresses, registers, fields, code tables,
  masks), no methods. Bring-up uses it to exercise ``RegisterI2CBase`` directly against
  the part, so a failure there cannot be driver logic.
* ``TMP102``: the driver, adding interpretation on top.

Data format, the part that bites: temperature, T_HIGH and T_LOW are *left-justified*
two's complement, 12 bits (normal) or 13 bits (extended mode, EM=1), 0.0625 C/LSB. Shift
the padding out **before** sign extension. Datasheet Tables 6-2/6-3 print the
right-justified count in their HEX column (50 C as ``320``); the wire word is in Tables
6-8/6-9 (``0x3200``).

Generated by ``dev_notebooks/drivers_tmp102_devNB``: edit the notebook, not this file.
"""

"TI TMP102 digital temperature sensor (SBOS397I).\n\nUsage on the board::\n\n    from micropython_i2c import I2CConnection\n    from micropython_i2c.drivers.tmp102 import TMP102\n\n    tmp = TMP102(I2CConnection(bus, 0x48, lock=bus_lock))\n    print(tmp.temperature())        # -> 23.4375\n\nTwo classes:\n\n* ``TMP102Regs``: datasheet facts only (addresses, registers, fields, code tables,\n  masks), no methods. Bring-up uses it to exercise ``RegisterI2CBase`` directly against\n  the part, so a failure there cannot be driver logic.\n* ``TMP102``: the driver, adding interpretation on top.\n\nData format, the part that bites: temperature, T_HIGH and T_LOW are *left-justified*\ntwo's complement, 12 bits (normal) or 13 bits (extended mode, EM=1), 0.0625 C/LSB. Shift\nthe padding out **before** sign extension. Datasheet Tables 6-2/6-3 print the\nright-justified count in their HEX column (50 C as ``320``); the wire word is in Tables\n6-8/6-9 (``0x3200``).\n\nGenerated by ``dev_notebooks/driver

# Imports
## External
None: the driver runs on MicroPython and uses only this package.

## Internal
Timing comes from `compat` so the same code runs on the board and on CPython.

In [2]:
%%extract ../src/micropython_i2c/drivers/tmp102.py -a
from micropython_i2c.compat import sleep_ms, ticks_add, ticks_diff, ticks_ms
from micropython_i2c.node import I2CDevice
from micropython_i2c.protocol import RegisterI2CBase

# Setup

Notebook-only: run the test cells in the notebook. `raise_on_error=True` makes a failing test
fail the run.

In [3]:
import ipytest

ipytest.autoconfig(raise_on_error=True)

# Test file
The test file header: imports only. It is written, not run.

In [4]:
%%writefile ../tests/test_tmp102.py
import pytest

from micropython_i2c import I2CConnection
from micropython_i2c.drivers.tmp102 import TMP102, TMP102Regs
from micropython_i2c.fakes import FakeBus, FakeTMP102

Overwriting ../tests/test_tmp102.py


Notebook-only: the imports the validation and test cells use. The classes under test are the
ones defined in this notebook, not the module's.

In [5]:
import pytest

from micropython_i2c import I2CConnection
from micropython_i2c.fakes import FakeBus, FakeTMP102

# Intent
**Purpose:** talk to a TI TMP102 temperature sensor in degrees Celsius: read it, set its
ALERT thresholds and configuration, and take one-shot readings.

**Why it exists:** the TMP102's words are left-justified two's complement whose width depends
on a config bit (extended mode), and its config register has three kinds of bits that cannot
read back as written. Getting either wrong gives numbers that look plausible and are wrong.

**Used by:** the routing table and manager (`"driver": "TMP102"`), bring-up notebooks, and the
bench (TMP102 on mux channel 2).

**Not its job:** reaching the part (the `I2CConnection` does addressing, mux routing and
locking) or register plumbing (`RegisterI2CBase` does caching and verified writes).

```python
tmp = TMP102(I2CConnection(bus, 0x48, [[0x70, 2]], lock))
tmp.temperature()                 # 23.4375
tmp.limits_set(20.0, 30.0)        # (20.0, 30.0)
```

```mermaid
classDiagram
    RegisterI2CBase <|-- TMP102Regs
    I2CDevice <|-- TMP102Regs
    TMP102Regs <|-- TMP102
    class TMP102Regs { datasheet facts only }
    class TMP102 { temperature(), limits, config properties, one_shot() }
```

**Data format** (temperature, T_HIGH, T_LOW):

| Mode | Value bits | Padding | Range | Wire word for 50 C |
|---|---|---|---|---|
| normal (EM=0) | top 12 | low 4 read 0 | -55 to +128 C | `0x3200` |
| extended (EM=1) | top 13 | low 3 read 0 | -55 to +150 C | `0x1900` |

One LSB is $0.0625\,^\circ\mathrm{C}$ in both modes:
$T = \mathrm{sign\_extend}(w \gg s,\ b) \times 0.0625$, with $(s, b) = (4, 12)$ or $(3, 13)$.

## What the module defines

| Name | What it is |
|---|---|
| `TMP102Regs` | datasheet facts: addresses, registers, fields, verify masks, POR values, code tables |
| `TMP102` | the driver: temperature, config properties, thresholds, one-shot |

# Code
## TODO:
Nothing

## Public names

In [6]:
%%extract ../src/micropython_i2c/drivers/tmp102.py -a
__all__ = ["TMP102", "TMP102Regs"]

## The class hierarchy
`TMP102Regs` mixes `RegisterI2CBase` (register protocol) into `I2CDevice` (transport through a
connection) and adds only class attributes. `TMP102` derives from it and adds the methods. Bring
up uses `TMP102Regs` on its own, so a failure there is the mixin, the connection or the wiring,
never driver logic.

## `TMP102Regs`
Every number here is from TI SBOS397I. The config register's verify mask leaves out the bits
that cannot read back as written: OS reads inverted, R1/R0 (resolution) are read-only and
always `11`, AL follows the comparator. The threshold registers have no such bits, so they
have no mask entry and verify on every bit.

In [7]:
%%extract ../src/micropython_i2c/drivers/tmp102.py -a
class TMP102Regs(RegisterI2CBase, I2CDevice):
    """Datasheet facts for the TMP102, with no driver logic."""

    ALLOWED_ADDRS = (0x48, 0x49, 0x4A, 0x4B)  # Table 6-4: ADD0 -> GND/V+/SDA/SCL
    BAUD_NOMINAL = 100_000
    BAUD_MAX = 400_000  # fast mode; HS mode needs a master code we cannot send
    BAUD_MIN = 10_000  # spec floor is ~1 kHz (SCL timeout); margin here

    BIG_ENDIAN = True
    REG_BYTES = 2

    # Pointer register values (Table 6-7). The pointer write is the same wire sequence
    # as a register read, so readfrom_mem works unmodified.
    REG_TEMP = 0x00  # read-only
    REG_CONFIG = 0x01
    REG_TLOW = 0x02
    REG_THIGH = 0x03

    # Config fields as (shift, width); byte 1 is the MSB (Tables 6-10/6-11).
    F_OS = (15, 1)  # one-shot / conversion ready
    F_RES = (13, 2)  # resolution, read-only, always 11
    F_FQ = (11, 2)  # fault queue
    F_POL = (10, 1)  # ALERT polarity
    F_TM = (9, 1)  # thermostat mode: 0 comparator, 1 interrupt
    F_SD = (8, 1)  # shutdown
    F_CR = (6, 2)  # conversion rate
    F_AL = (5, 1)  # alert status, read-only
    F_EM = (4, 1)  # extended mode

    # Config bits that cannot read back as written: OS reads inverted, R1/R0 are
    # read-only, AL is comparator-driven. The threshold registers have no such bits,
    # so they verify on every bit (no entry needed).
    CONFIG_VERIFY_MASK = 0xFFFF & ~(0x8000 | 0x6000 | 0x0020)
    VERIFY_MASKS = {REG_CONFIG: CONFIG_VERIFY_MASK}

    CFG_POR = 0x60A0
    THIGH_POR = 0x5000  # 80 C
    TLOW_POR = 0x4B00  # 75 C

    LSB_C = 0.0625
    TEMP_MIN = -55.0
    TEMP_MAX = 150.0  # extended-mode ceiling

    CONV_RATES = {0: 0.25, 1: 1.0, 2: 4.0, 3: 8.0}  # Hz, Table 6-5
    FAULT_QUEUE = {0: 1, 1: 2, 2: 4, 3: 6}  # Table 6-12
    CONV_TIME_MS = 26  # single conversion, typical (section 6.5; measured 29 ms incl. I2C)
    CONV_TIME_MAX_MS = 35  # single conversion, maximum

### validation/test
A helper that puts a `FakeTMP102` (power-on state, 25 C) on a fake bus and binds a class to it.
It is shared by the validation cells and the test file.

In [8]:
%%extract ../tests/test_tmp102.py -a
def on_fake(cls, temp_word=0x1900):
    """Bind ``cls`` to a ``FakeTMP102`` at 0x48 on a fresh fake bus (25 C by default)."""
    bus = FakeBus()
    bus.attach(0x48, FakeTMP102(temp_word=temp_word))
    return cls(I2CConnection(bus, 0x48))

#### validation: nominal usage
The config register at power-on, decoded field by field, and the verify mask built from the
fields rather than trusted as a hex literal.

In [9]:
regs = on_fake(TMP102Regs)
cfg = regs.reg_read(regs.REG_CONFIG)
print("config at POR:", hex(cfg), "| matches CFG_POR:", cfg == regs.CFG_POR)
for name in ("F_OS", "F_RES", "F_FQ", "F_POL", "F_TM", "F_SD", "F_CR", "F_AL", "F_EM"):
    print("  {:<6} = {}".format(name, regs.field_extract(cfg, getattr(regs, name))))
excluded = regs.field_mask(regs.F_OS) | regs.field_mask(regs.F_RES) | regs.field_mask(regs.F_AL)
print("verify mask excludes exactly OS, RES, AL:", 0xFFFF & ~regs.CONFIG_VERIFY_MASK == excluded)
print("thresholds verify on every bit:", regs.verify_mask_for(regs.REG_THIGH) is None)

config at POR: 0x60a0 | matches CFG_POR: True
  F_OS   = 0
  F_RES  = 3
  F_FQ   = 0
  F_POL  = 0
  F_TM   = 0
  F_SD   = 0
  F_CR   = 2
  F_AL   = 1
  F_EM   = 0
verify mask excludes exactly OS, RES, AL: True
thresholds verify on every bit: True


#### validation: anti-nominal usage
An exact verify of the config register must fail: R1/R0 always read `11`, so a request with
them cleared never reads back as written. The class mask is what makes the same write pass.

In [10]:
regs = on_fake(TMP102Regs)
sabotaged = regs.CFG_POR & ~regs.field_mask(regs.F_RES)
try:
    regs.reg_set(regs.REG_CONFIG, sabotaged, verify_mask=0xFFFF)  # R1/R0 are read-only
except ValueError as err:
    print(f"raised ValueError, as expected:\n    {err}")
else:
    raise AssertionError("an exact verify of a cleared R1/R0 should have raised ValueError")
finally:
    print("same write under the class mask lands:", hex(regs.reg_set(regs.REG_CONFIG, sabotaged)))

raised ValueError, as expected:
    reg 0x01: wrote 0xa0, device holds 0x60a0 (mask 0xffff)
same write under the class mask lands: 0x60a0


#### unit

The nine config fields do not overlap and cover every data bit (bits 15..4).

In [11]:
%%extract ../tests/test_tmp102.py -a --strip-ipytest
%%ipytest


def test_fields_tile_the_data_bits():
    covered = 0
    for name in ("F_OS", "F_RES", "F_FQ", "F_POL", "F_TM", "F_SD", "F_CR", "F_AL", "F_EM"):
        mask = TMP102Regs.field_mask(getattr(TMP102Regs, name))
        assert covered & mask == 0
        covered |= mask
    assert covered == 0xFFF0  # the low nibble is reserved

.                                                                                            [100%]
========================================= warnings summary =========================================
../.venv/lib/python3.13/site-packages/anyio/pytest_plugin.py:14
  /home/nonyourbusiness/Insync/string12oil@gmail.com/Google Drive/Project_restart/micropython-i2c-toolkit/.venv/lib/python3.13/site-packages/anyio/pytest_plugin.py:14: PytestRemovedIn10Warning: _pytest.python.CallSpec2 has been renamed to CallSpec.
  The CallSpec2 alias will be removed in pytest 10.
  Update imports to use CallSpec instead.
  See https://docs.pytest.org/en/stable/deprecations.html#callspec2-renamed
    from _pytest.python import CallSpec2

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
1 passed, 1 warning in 0.02s


The config verify mask leaves out exactly OS, R1/R0 and AL, and only the config register has a mask.

In [12]:
%%extract ../tests/test_tmp102.py -a --strip-ipytest
%%ipytest


def test_config_verify_mask_excludes_os_res_al():
    m = TMP102Regs.field_mask
    excluded = m(TMP102Regs.F_OS) | m(TMP102Regs.F_RES) | m(TMP102Regs.F_AL)
    assert 0xFFFF & ~TMP102Regs.CONFIG_VERIFY_MASK == excluded
    assert TMP102Regs.VERIFY_MASKS == {TMP102Regs.REG_CONFIG: TMP102Regs.CONFIG_VERIFY_MASK}

.                                                                                            [100%]
1 passed in 0.02s


The power-on config word decodes to R1/R0 = `11`, a 4 Hz conversion rate and AL = 1.

In [13]:
%%extract ../tests/test_tmp102.py -a --strip-ipytest
%%ipytest


def test_por_values_decode_as_the_datasheet_says():
    x = TMP102Regs.field_extract
    assert x(TMP102Regs.CFG_POR, TMP102Regs.F_RES) == 0b11
    assert TMP102Regs.CONV_RATES[x(TMP102Regs.CFG_POR, TMP102Regs.F_CR)] == 4.0
    assert x(TMP102Regs.CFG_POR, TMP102Regs.F_AL) == 1

.                                                                                            [100%]
1 passed in 0.03s


A write with R1/R0 cleared fails an exact verify and lands under the class mask.

In [14]:
%%extract ../tests/test_tmp102.py -a --strip-ipytest
%%ipytest


def test_masked_write_lands_and_exact_write_is_refused():
    regs = on_fake(TMP102Regs)
    sabotaged = regs.CFG_POR & ~regs.field_mask(regs.F_RES)
    with pytest.raises(ValueError, match="mask 0xffff"):
        regs.reg_set(regs.REG_CONFIG, sabotaged, verify_mask=0xFFFF)
    assert regs.reg_set(regs.REG_CONFIG, sabotaged) == regs.CFG_POR

.                                                                                            [100%]
1 passed in 0.04s


The four ADD0 addresses are allowed, others are not, and 1 MHz is above the part's limit.

In [15]:
%%extract ../tests/test_tmp102.py -a --strip-ipytest
%%ipytest


def test_address_and_clock_limits():
    assert all(TMP102Regs.addr_ok(a) for a in (0x48, 0x49, 0x4A, 0x4B))
    assert not TMP102Regs.addr_ok(0x70)
    assert TMP102Regs.baud_ok(100_000) and not TMP102Regs.baud_ok(1_000_000)

.                                                                                            [100%]
1 passed in 0.03s


## `TMP102`
Interpretation on top of `TMP102Regs`: degrees in and out, named config settings, thresholds
and one-shot.

Extended mode changes how every temperature word decodes, so the driver caches it in `em` and
resyncs it on construction and after every config write. A stale flag is silent: a 13-bit word
decoded as 12-bit reads exactly half the true value.

One-shot is the one place the driver writes raw: OS is a handshake, excluded from the verify
mask, so a managed write cannot check it.

In [16]:
%%extract ../src/micropython_i2c/drivers/tmp102.py -a
class TMP102(TMP102Regs):
    """TMP102 driver: temperature in C, config properties, thresholds, one-shot.

    Extended mode changes how every temperature word decodes, so it is cached in
    ``self.em`` and resynced on construction and after every config write. A stale flag
    is silent: a 13-bit word decoded as 12-bit reads exactly half the true value.
    ``mode_refresh()`` resyncs after something outside the driver touched the part.

    Changing EM has a second trap, measured on the part: D0 of the temperature word follows
    EM at once, but the data keeps the old format until the next conversion (61 ms at 4 Hz).
    Decoding it in the new format reads half or double the true value. So after a write
    that changes EM the driver waits one conversion period plus the maximum conversion time
    before returning (about 285 ms at 4 Hz). In shutdown nothing converts until a one-shot,
    so ``temperature()`` refuses to decode until ``one_shot()`` has run.

    The TMP102 has no ID register. The strongest identification available is the
    read-only R1/R0 bits, which always read ``11``.

    Attributes:
        em: True in extended (13-bit) mode, as last read from the part.
        format_stale: True after EM changed in shutdown, until the next ``one_shot()``.
    """

    def __init__(self, conn, **kwargs):
        super().__init__(conn, **kwargs)
        self.em = False
        self.format_stale = False
        self.mode_refresh()

    # Codecs: classmethods with an explicit ``em`` so they test without a device.

    @staticmethod
    def _format(em):
        """(shift, bits) of the left-justified word in the given mode."""
        return (3, 13) if em else (4, 12)

    @classmethod
    def celsius_from_raw(cls, word, em):
        """Decode a left-justified two's-complement word into degrees C."""
        shift, bits = cls._format(em)
        return cls.sign_extend(word >> shift, bits) * cls.LSB_C

    @classmethod
    def celsius_to_raw(cls, celsius, em):
        """Encode degrees C into a left-justified word, rounding half away from zero.

        Two bounds apply: the register width (what can be represented) and the part's
        operating range (what is meaningful). In extended mode the second is tighter,
        so checking only width would accept a 200 C threshold the part can never reach.

        Raises:
            ValueError: If either bound is exceeded.
        """
        shift, bits = cls._format(em)
        counts = int(celsius / cls.LSB_C + (0.5 if celsius >= 0 else -0.5))
        lo, hi = -(1 << (bits - 1)), (1 << (bits - 1)) - 1
        if not lo <= counts <= hi:
            raise ValueError(
                "{} C not representable in {}-bit mode ({:.4f} to {:.4f} C)".format(
                    celsius, bits, lo * cls.LSB_C, hi * cls.LSB_C
                )
            )
        if not cls.TEMP_MIN <= celsius <= cls.TEMP_MAX:
            raise ValueError(
                "{} C outside the operating range ({} to {} C)".format(
                    celsius, cls.TEMP_MIN, cls.TEMP_MAX
                )
            )
        return (counts & ((1 << bits) - 1)) << shift

    # Temperature. Raw reads only: the temperature register must never be cached.

    def temperature(self):
        """Most recent conversion in degrees C.

        Raises:
            RuntimeError: If EM changed while in shutdown and no one-shot has run since: the
                register still holds a word in the old format.
        """
        if self.format_stale:
            raise RuntimeError(
                "extended mode changed in shutdown: the temperature register keeps the old "
                "format until the next conversion; take a one_shot() first"
            )
        return self.celsius_from_raw(self.reg_read(self.REG_TEMP), self.em)

    def temperature_raw(self):
        """The undecoded 16-bit temperature word."""
        return self.reg_read(self.REG_TEMP)

    # Configuration.

    def config_get(self, refresh=True):
        """The config register. Fresh by default: OS and AL are live status bits."""
        return self.reg_get(self.REG_CONFIG, refresh=refresh)

    def _em_resync(self):
        """Update ``em`` from the confirmed config value; if it changed, settle the format."""
        cfg = self.cached(self.REG_CONFIG)
        old, self.em = self.em, bool(self.field_extract(cfg, self.F_EM))
        if self.em == old:
            return
        if self.field_extract(cfg, self.F_SD):
            self.format_stale = True  # no conversion until a one-shot
            return
        self.format_stale = False
        rate = self.CONV_RATES[self.field_extract(cfg, self.F_CR)]
        sleep_ms(int(1000 / rate) + self.CONV_TIME_MAX_MS)  # the next conversion has run

    def config_set(self, value):
        """Write the whole config register (verified) and resync EM."""
        result = self.reg_set(self.REG_CONFIG, value)
        self._em_resync()
        return result

    def config_field_set(self, field, value):
        """Write one config field (verified) and resync EM."""
        result = self.field_set(self.REG_CONFIG, field, value)
        self._em_resync()
        return result

    def config_reset(self):
        """Restore the config register to its power-on value."""
        return self.config_set(self.CFG_POR)

    def mode_refresh(self):
        """Re-read EM from the device. Returns the new value."""
        self.em = self.flag_get(self.REG_CONFIG, self.F_EM)
        return self.em

    @property
    def extended_mode(self):
        """13-bit mode, allowing readings above 128 C.

        Setting it waits for the data to take the new format (see the class docstring). The
        thresholds are stored as raw words, so their meaning changes with EM: set the limits
        again after changing it.
        """
        return self.em

    @extended_mode.setter
    def extended_mode(self, enable):
        self.config_field_set(self.F_EM, 1 if enable else 0)

    @property
    def shutdown(self):
        """Shutdown mode; halts conversions. Required before ``one_shot()``."""
        return self.flag_get(self.REG_CONFIG, self.F_SD)

    @shutdown.setter
    def shutdown(self, enable):
        self.config_field_set(self.F_SD, 1 if enable else 0)

    @property
    def conversion_rate(self):
        """Continuous conversion rate in Hz: 0.25, 1, 4 (default) or 8."""
        return self.coded_get(self.REG_CONFIG, self.F_CR, self.CONV_RATES)

    @conversion_rate.setter
    def conversion_rate(self, hz):
        self.coded_set(self.REG_CONFIG, self.F_CR, self.CONV_RATES, hz)

    @property
    def fault_queue(self):
        """Consecutive out-of-limit conversions before ALERT: 1, 2, 4 or 6."""
        return self.coded_get(self.REG_CONFIG, self.F_FQ, self.FAULT_QUEUE)

    @fault_queue.setter
    def fault_queue(self, faults):
        self.coded_set(self.REG_CONFIG, self.F_FQ, self.FAULT_QUEUE, faults)

    @property
    def thermostat_mode(self):
        """False = comparator mode, True = interrupt mode (ALERT latches)."""
        return self.flag_get(self.REG_CONFIG, self.F_TM)

    @thermostat_mode.setter
    def thermostat_mode(self, interrupt):
        self.config_field_set(self.F_TM, 1 if interrupt else 0)

    @property
    def alert_polarity_high(self):
        """False (default) = ALERT active low, True = active high."""
        return self.flag_get(self.REG_CONFIG, self.F_POL)

    @alert_polarity_high.setter
    def alert_polarity_high(self, active_high):
        self.config_field_set(self.F_POL, 1 if active_high else 0)

    @property
    def alert(self):
        """Raw AL bit. With POL=0 it reads 1 when NOT alerting; POL inverts this."""
        return self.flag_get(self.REG_CONFIG, self.F_AL)

    @property
    def conversion_ready(self):
        """OS bit: True when idle. Only meaningful in shutdown mode."""
        return self.flag_get(self.REG_CONFIG, self.F_OS)

    # Thresholds. Same format as the temperature register.

    def high_limit_get(self):
        """T_HIGH in degrees C (power-on 80 C)."""
        return self.celsius_from_raw(self.reg_read(self.REG_THIGH), self.em)

    def low_limit_get(self):
        """T_LOW in degrees C (power-on 75 C)."""
        return self.celsius_from_raw(self.reg_read(self.REG_TLOW), self.em)

    def _limit_set(self, reg, celsius):
        raw = self.celsius_to_raw(celsius, self.em)
        return self.celsius_from_raw(self.reg_set(reg, raw), self.em)

    def high_limit_set(self, celsius):
        """Set T_HIGH, verified exactly. Returns what landed (within half an LSB)."""
        return self._limit_set(self.REG_THIGH, celsius)

    def low_limit_set(self, celsius):
        """Set T_LOW, verified exactly. Keep it below T_HIGH; the part does not check."""
        return self._limit_set(self.REG_TLOW, celsius)

    def limits_set(self, low_c, high_c):
        """Set both thresholds, rejecting a crossed pair before any bus traffic."""
        if low_c >= high_c:
            raise ValueError("T_LOW ({}) must be below T_HIGH ({})".format(low_c, high_c))
        return self.low_limit_set(low_c), self.high_limit_set(high_c)

    def limits_reset(self):
        """Restore both thresholds to 75 C / 80 C."""
        self.reg_set(self.REG_TLOW, self.TLOW_POR)
        self.reg_set(self.REG_THIGH, self.THIGH_POR)

    # One-shot.

    def one_shot(self, timeout_ms=100):
        """Trigger one conversion from shutdown and return it in degrees C.

        Handshake: write OS=1, poll until OS reads 1 again, read the temperature. OS is
        excluded from verification, so this uses the raw write path.

        Raises:
            RuntimeError: If not in shutdown mode (a continuous-mode reading would be
                indistinguishable from a real one-shot), or on timeout.
        """
        cfg = self.config_get()
        if not cfg & self.field_mask(self.F_SD):
            raise RuntimeError("one_shot() requires shutdown mode; set .shutdown = True")

        os_bit = self.field_mask(self.F_OS)
        self.reg_write(self.REG_CONFIG, cfg | os_bit)
        deadline = ticks_add(ticks_ms(), timeout_ms)
        while not self.reg_read(self.REG_CONFIG) & os_bit:
            if ticks_diff(deadline, ticks_ms()) <= 0:
                raise RuntimeError("conversion timed out after {} ms".format(timeout_ms))
            sleep_ms(1)
        self.format_stale = False  # this conversion used the current format
        return self.temperature()

### validation/test: codecs
The datasheet's wire words, both ways. Tables 6-2/6-3 print a right-justified count in their
HEX column (50 C as `320`); the wire word is that count shifted left by 4 (normal) or 3
(extended), as Tables 6-8/6-9 show. The expected words below are wire words.

The datasheet cases, shared by the validation cell below and the unit tests (extracted to the
test file).

In [17]:
%%extract ../tests/test_tmp102.py -a
CASES_12 = [(127.9375, 0x7FF0), (100.0, 0x6400), (75.0, 0x4B00), (50.0, 0x3200),
            (25.0, 0x1900), (0.25, 0x0040), (0.0, 0x0000), (-0.25, 0xFFC0),
            (-25.0, 0xE700), (-55.0, 0xC900)]  # fmt: skip
CASES_13 = [(150.0, 0x4B00), (128.0, 0x4000), (100.0, 0x3200), (25.0, 0x0C80),
            (0.0, 0x0000), (-25.0, 0xF380), (-55.0, 0xE480)]  # fmt: skip

#### validation: nominal usage

In [18]:
for em, cases in ((False, CASES_12), (True, CASES_13)):
    print("{}-bit mode".format(13 if em else 12))
    for celsius, word in cases:
        enc, dec = TMP102.celsius_to_raw(celsius, em), TMP102.celsius_from_raw(word, em)
        print("  {:>9} C -> {:#06x} (want {:#06x}) -> {} C".format(celsius, enc, word, dec))

12-bit mode
   127.9375 C -> 0x7ff0 (want 0x7ff0) -> 127.9375 C
      100.0 C -> 0x6400 (want 0x6400) -> 100.0 C
       75.0 C -> 0x4b00 (want 0x4b00) -> 75.0 C
       50.0 C -> 0x3200 (want 0x3200) -> 50.0 C
       25.0 C -> 0x1900 (want 0x1900) -> 25.0 C
       0.25 C -> 0x0040 (want 0x0040) -> 0.25 C
        0.0 C -> 0x0000 (want 0x0000) -> 0.0 C
      -0.25 C -> 0xffc0 (want 0xffc0) -> -0.25 C
      -25.0 C -> 0xe700 (want 0xe700) -> -25.0 C
      -55.0 C -> 0xc900 (want 0xc900) -> -55.0 C
13-bit mode
      150.0 C -> 0x4b00 (want 0x4b00) -> 150.0 C
      128.0 C -> 0x4000 (want 0x4000) -> 128.0 C
      100.0 C -> 0x3200 (want 0x3200) -> 100.0 C
       25.0 C -> 0x0c80 (want 0x0c80) -> 25.0 C
        0.0 C -> 0x0000 (want 0x0000) -> 0.0 C
      -25.0 C -> 0xf380 (want 0xf380) -> -25.0 C
      -55.0 C -> 0xe480 (want 0xe480) -> -55.0 C


#### validation: anti-nominal usage
Two bounds, two failures. 200 C does not fit 12 bits; in 13 bits it fits the register but is
above the part's 150 C ceiling, which is the bound a width check alone would miss.

In [19]:
try:
    TMP102.celsius_to_raw(200.0, False)  # wider than 12 bits can hold
except ValueError as err:
    print(f"raised ValueError, as expected:\n    {err}")
else:
    raise AssertionError("200 C in 12-bit mode should have raised ValueError")

raised ValueError, as expected:
    200.0 C not representable in 12-bit mode (-128.0000 to 127.9375 C)


The second bound: 200 C fits 13 bits but is above the 150 C operating range.

In [20]:
try:
    TMP102.celsius_to_raw(200.0, True)  # fits 13 bits, but above the 150 C operating range
except ValueError as err:
    print(f"raised ValueError, as expected:\n    {err}")
else:
    raise AssertionError("200 C in 13-bit mode should have raised ValueError")

raised ValueError, as expected:
    200.0 C outside the operating range (-55.0 to 150.0 C)


#### unit

In [21]:
%%extract ../tests/test_tmp102.py -a --strip-ipytest
%%ipytest


@pytest.mark.parametrize("celsius, word", CASES_12)
def test_codec_12bit_matches_datasheet(celsius, word):
    assert TMP102.celsius_to_raw(celsius, False) == word
    assert TMP102.celsius_from_raw(word, False) == celsius

..........                                                                                   [100%]
10 passed in 0.08s


Extended mode: every 13-bit datasheet case encodes and decodes exactly.

In [22]:
%%extract ../tests/test_tmp102.py -a --strip-ipytest
%%ipytest


@pytest.mark.parametrize("celsius, word", CASES_13)
def test_codec_13bit_matches_datasheet(celsius, word):
    assert TMP102.celsius_to_raw(celsius, True) == word
    assert TMP102.celsius_from_raw(word, True) == celsius

.......                                                                                      [100%]
7 passed in 0.06s


Out-of-range temperatures are refused in both modes, by width or by operating range.

In [23]:
%%extract ../tests/test_tmp102.py -a --strip-ipytest
%%ipytest


@pytest.mark.parametrize("celsius, em", [(200.0, False), (200.0, True), (-100.0, False),
                                         (-100.0, True)])  # fmt: skip
def test_codec_refuses_out_of_range(celsius, em):
    with pytest.raises(ValueError, match="representable|operating range"):
        TMP102.celsius_to_raw(celsius, em)

....                                                                                         [100%]
4 passed in 0.05s


The decode shifts the padding out before sign extension; extending the raw word is wrong.

In [24]:
%%extract ../tests/test_tmp102.py -a --strip-ipytest
%%ipytest


def test_sign_extension_needs_the_shift_first():
    # 0xE700 is -25 C; extending the unshifted word would treat padding as magnitude.
    assert TMP102.celsius_from_raw(0xE700, False) == -25.0
    assert TMP102.sign_extend(0xE700, 12) * TMP102.LSB_C != -25.0

.                                                                                            [100%]
1 passed in 0.03s


### validation/test: the driver on a fake part
`FakeTMP102` keeps R1/R0 at `11` and AL read-only like the real part, so verified writes,
extended mode and the one-shot handshake behave as on silicon.

#### validation: nominal usage

In [25]:
tmp = on_fake(TMP102)
print("temperature:", tmp.temperature(), "C | extended mode:", tmp.extended_mode)
print("limits:", tmp.low_limit_get(), "/", tmp.high_limit_get(), "C")
print("set 20/30 C ->", tmp.limits_set(20.0, 30.0))
tmp.conversion_rate = 8.0
print("conversion rate:", tmp.conversion_rate, "Hz | fault queue:", tmp.fault_queue)
tmp.extended_mode = True
print(
    "extended mode on -> em:",
    tmp.em,
    "| still reads",
    tmp.temperature(),
    "C (the driver waited for the new format)",
)

temperature: 25.0 C | extended mode: False
limits: 75.0 / 80.0 C
set 20/30 C -> (20.0, 30.0)
conversion rate: 8.0 Hz | fault queue: 1
extended mode on -> em: True | still reads 25.0 C (the driver waited for the new format)


#### validation: anti-nominal usage
A crossed threshold pair is refused before any bus traffic, and one-shot refuses to run outside
shutdown mode, where it would return a stale continuous reading.

In [26]:
tmp = on_fake(TMP102)
try:
    tmp.limits_set(80.0, 20.0)  # T_LOW must be below T_HIGH
except ValueError as err:
    print(f"raised ValueError, as expected:\n    {err}")
else:
    raise AssertionError("crossed limits should have raised ValueError")
finally:
    print("thresholds untouched:", (tmp.low_limit_get(), tmp.high_limit_get()) == (75.0, 80.0))

raised ValueError, as expected:
    T_LOW (80.0) must be below T_HIGH (20.0)
thresholds untouched: True


One-shot outside shutdown mode would return a stale continuous reading, so it is refused.

In [27]:
tmp = on_fake(TMP102)
try:
    tmp.one_shot()  # the part is in continuous mode
except RuntimeError as err:
    print(f"raised RuntimeError, as expected:\n    {err}")
else:
    raise AssertionError("one_shot() outside shutdown mode should have raised RuntimeError")

raised RuntimeError, as expected:
    one_shot() requires shutdown mode; set .shutdown = True


A conversion rate the part cannot do is refused instead of writing a wrong code.

In [28]:
tmp = on_fake(TMP102)
try:
    tmp.conversion_rate = 2.0  # not one of 0.25, 1, 4, 8 Hz
except ValueError as err:
    print(f"raised ValueError, as expected:\n    {err}")
else:
    raise AssertionError("conversion_rate = 2.0 should have raised ValueError")

raised ValueError, as expected:
    2.0 not in [0.25, 1.0, 4.0, 8.0]


#### unit

In [29]:
%%extract ../tests/test_tmp102.py -a --strip-ipytest
%%ipytest


def test_reads_and_threshold_round_trip():
    tmp = on_fake(TMP102)
    assert tmp.temperature() == 25.0
    assert (tmp.low_limit_get(), tmp.high_limit_get()) == (75.0, 80.0)
    assert tmp.limits_set(20.0, 30.0) == (20.0, 30.0)
    assert abs(tmp.low_limit_set(20.03) - 20.03) <= tmp.LSB_C / 2
    tmp.limits_reset()
    assert tmp.high_limit_get() == 80.0

.                                                                                            [100%]
1 passed in 0.03s


A crossed threshold pair is refused before any bus traffic.

In [30]:
%%extract ../tests/test_tmp102.py -a --strip-ipytest
%%ipytest


def test_crossed_limits_are_refused():
    tmp = on_fake(TMP102)
    with pytest.raises(ValueError, match="must be below"):
        tmp.limits_set(80.0, 20.0)

.                                                                                            [100%]
1 passed in 0.03s


Every config property reads back what was set.

In [31]:
%%extract ../tests/test_tmp102.py -a --strip-ipytest
%%ipytest


def test_config_properties_round_trip():
    tmp = on_fake(TMP102)
    for hz in (0.25, 1.0, 8.0, 4.0):
        tmp.conversion_rate = hz
        assert tmp.conversion_rate == hz
    for n in (1, 2, 4, 6):
        tmp.fault_queue = n
        assert tmp.fault_queue == n
    for attr in ("thermostat_mode", "alert_polarity_high", "shutdown"):
        for state in (True, False):
            setattr(tmp, attr, state)
            assert getattr(tmp, attr) is state

.                                                                                            [100%]
1 passed in 0.03s


The driver tracks extended mode, and `mode_refresh()` corrects a stale flag.

In [32]:
%%extract ../tests/test_tmp102.py -a --strip-ipytest
%%ipytest


def test_extended_mode_is_tracked_and_refreshable():
    tmp = on_fake(TMP102)
    tmp.extended_mode = True
    assert tmp.em is True
    tmp.em = False  # deliberately stale
    assert tmp.mode_refresh() is True
    tmp.extended_mode = False
    assert tmp.em is False

.                                                                                            [100%]
1 passed in 0.60s


Right after switching extended mode on or off, the temperature reads the same: the driver
waits for the part to reformat the data instead of decoding the old word in the new format.

In [33]:
%%extract ../tests/test_tmp102.py -a --strip-ipytest
%%ipytest


def test_extended_mode_switch_waits_for_the_new_format():
    tmp = on_fake(TMP102)
    before = tmp.temperature()
    tmp.extended_mode = True
    assert tmp.temperature() == before
    tmp.extended_mode = False
    assert tmp.temperature() == before

.                                                                                            [100%]
1 passed in 0.60s


In shutdown nothing converts, so after an EM change the driver refuses to decode until a
one-shot has run in the new format.

In [34]:
%%extract ../tests/test_tmp102.py -a --strip-ipytest
%%ipytest


def test_extended_mode_switch_in_shutdown_needs_a_one_shot():
    tmp = on_fake(TMP102)
    tmp.shutdown = True
    tmp.extended_mode = True
    with pytest.raises(RuntimeError, match="one_shot"):
        tmp.temperature()
    assert tmp.one_shot() == 25.0
    assert tmp.temperature() == 25.0

.                                                                                            [100%]
1 passed in 0.03s


One-shot needs shutdown mode, and in shutdown it returns a reading.

In [35]:
%%extract ../tests/test_tmp102.py -a --strip-ipytest
%%ipytest


def test_one_shot_needs_shutdown_mode():
    tmp = on_fake(TMP102)
    with pytest.raises(RuntimeError, match="shutdown"):
        tmp.one_shot()
    tmp.shutdown = True
    assert tmp.one_shot() == 25.0

.                                                                                            [100%]
1 passed in 0.03s


# Hardware validation

> ## ⚠️ STOP — do not run this section unattended
>
> **Everything below talks to the real board.** It connects to the FeatherS3, uploads the
> package, rewrites the TMP102's configuration and threshold registers, and has steps where
> **you** must act (pinch the sensor) or keep your hands off it.
>
> - Do **not** use *Run All*, *Run All Below* or `jlx run` on this section and walk away.
> - Run it **one cell at a time**. Read the **What to do** line above each cell first.
> - Each cell puts the sensor back to its power-on state when it finishes, even if it fails.
> - The section ends with `%board disconnect`. If you stop partway, run that cell yourself.

Hardware (as found by `drivers_tca9548a_devNB`, 2026-11-13): FeatherS3, SparkFun Qwiic Mux
(TCA9548A) at `0x70` (address jumpers open) on **bus 1** (scl=15, sda=16), TMP102 at `0x48` on
mux **channel 2**. The TMAG5273 (`0x22`) is on channel 0; nothing on the bench shares the mux's
address.

Layout: connect and set up, then the mux and the layer stack, then **matching tests** (one per
unit test above, same name, run on the real part and on MicroPython), then hardware-only
checks, then the tally and disconnect. Every check prints a `PASS`/`FAIL` line and counts
towards the tally; the last cell fails the run if any check failed.

## Connect and set up

Notebook-only: connect to the board. With one USB board `%board connect` finds it; with
several, give the port.

**What to do:** make sure the board is plugged in and no other notebook owns it.

In [36]:
%board connect

Connected /dev/ttyACM0: FeatherS3 with ESP32-S3 - micropython 1.28.0 (v1.28.0 on 2026-04-06)


Notebook-only: remove the package an earlier upload left on the board, and make sure `/lib`
exists. Copying into an existing `/lib/micropython_i2c` would put the new package *inside* the
old one (`cp` rules), and old modules would keep loading.

**What to do:** nothing.

In [37]:
import os


def rmtree(path):
    """Delete a board folder and everything in it."""
    for entry in os.ilistdir(path):
        child = path + "/" + entry[0]
        if entry[1] == 0x4000:  # folder
            rmtree(child)
        else:
            os.remove(child)
    os.rmdir(path)


try:
    rmtree("/lib/micropython_i2c")
    print("removed the old /lib/micropython_i2c")
except OSError:
    print("no /lib/micropython_i2c on the board yet")
try:
    os.mkdir("/lib")
except OSError:
    pass  # already there

removed the old /lib/micropython_i2c


Notebook-only: copy `src/micropython_i2c`, including the file this notebook just wrote, to
`/lib/micropython_i2c` on the board (the cell above removed any old copy, so it lands exactly
there). The copy runs on the host; the path is relative to `dev_notebooks/`. Host
`__pycache__` folders are not copied.

**What to do:** nothing. Takes a few seconds.

In [38]:
%%host
from pathlib import Path

pkg = next(p for p in (Path("../src"), Path("src")) if p.is_dir()) / "micropython_i2c"
%files copy -r {pkg} :/lib/micropython_i2c

copied ../src/micropython_i2c/__init__.py -> :/lib/micropython_i2c/__init__.py
copied ../src/micropython_i2c/bits.py -> :/lib/micropython_i2c/bits.py
copied ../src/micropython_i2c/boards.py -> :/lib/micropython_i2c/boards.py
copied ../src/micropython_i2c/compat.py -> :/lib/micropython_i2c/compat.py
copied ../src/micropython_i2c/connection.py -> :/lib/micropython_i2c/connection.py
copied ../src/micropython_i2c/crc.py -> :/lib/micropython_i2c/crc.py
copied ../src/micropython_i2c/debug.py -> :/lib/micropython_i2c/debug.py
copied ../src/micropython_i2c/drivers/__init__.py -> :/lib/micropython_i2c/drivers/__init__.py
copied ../src/micropython_i2c/drivers/qwiic_relay.py -> :/lib/micropython_i2c/drivers/qwiic_relay.py
copied ../src/micropython_i2c/drivers/shtc3.py -> :/lib/micropython_i2c/drivers/shtc3.py
copied ../src/micropython_i2c/drivers/tca9548a.py -> :/lib/micropython_i2c/drivers/tca9548a.py
copied ../src/micropython_i2c/drivers/tmag5273.py -> :/lib/micropython_i2c/drivers/tmag5273.py


['../src/micropython_i2c/__init__.py -> :/lib/micropython_i2c/__init__.py',
 '../src/micropython_i2c/bits.py -> :/lib/micropython_i2c/bits.py',
 '../src/micropython_i2c/boards.py -> :/lib/micropython_i2c/boards.py',
 '../src/micropython_i2c/compat.py -> :/lib/micropython_i2c/compat.py',
 '../src/micropython_i2c/connection.py -> :/lib/micropython_i2c/connection.py',
 '../src/micropython_i2c/crc.py -> :/lib/micropython_i2c/crc.py',
 '../src/micropython_i2c/debug.py -> :/lib/micropython_i2c/debug.py',
 '../src/micropython_i2c/drivers/__init__.py -> :/lib/micropython_i2c/drivers/__init__.py',
 '../src/micropython_i2c/drivers/qwiic_relay.py -> :/lib/micropython_i2c/drivers/qwiic_relay.py',
 '../src/micropython_i2c/drivers/shtc3.py -> :/lib/micropython_i2c/drivers/shtc3.py',
 '../src/micropython_i2c/drivers/tca9548a.py -> :/lib/micropython_i2c/drivers/tca9548a.py',
 '../src/micropython_i2c/drivers/tmag5273.py -> :/lib/micropython_i2c/drivers/tmag5273.py',
 '../src/micropython_i2c/drivers/tmp

Notebook-only: drop any modules an earlier run left on the board, so the new files load, then
import what the checks use and zero the tally.

**What to do:** nothing.

In [39]:
import sys

for name in [m for m in sys.modules if m.startswith("micropython_i2c")]:
    del sys.modules[name]

from micropython_i2c import I2CConnection, default_lock
from micropython_i2c.boards import bus_open
from micropython_i2c.compat import sleep_ms, ticks_diff, ticks_ms
from micropython_i2c.debug import check, check_raises, check_reset, check_summary, show
from micropython_i2c.debug import check_counts  # noqa: F401  (read from the host with %eval)
from micropython_i2c.drivers.tca9548a import TCA9548A
from micropython_i2c.drivers.tmp102 import TMP102, TMP102Regs

check_reset()

Notebook-only: the bus, the mux, and helpers that give each check a fresh object on a part at
power-on values, and put it back afterwards. If the bench is rewired, change `MUX_BUS` /
`MUX_ADDR` here (`drivers_tca9548a_devNB` finds them).

**What to do:** nothing.

In [40]:
MUX_BUS = 1  # second Qwiic chain, scl=15 sda=16
MUX_ADDR = 0x70  # SparkFun default, jumpers open (fine: no 0x70 part behind the mux)
TMP_ADDR = 0x48  # ADD0 -> GND
TMP_CHANNEL = 2
TMP_PATH = [[MUX_ADDR, TMP_CHANNEL]]

bus = bus_open("feathers3", MUX_BUS, freq=100_000)
bus_lock = default_lock()
mux = TCA9548A(bus, MUX_ADDR, lock=bus_lock)


def restore(dev):
    """Put the real part back to power-on: config, both thresholds, empty cache."""
    dev.reg_set(dev.REG_CONFIG, dev.CFG_POR)
    dev.reg_set(dev.REG_THIGH, dev.THIGH_POR)
    dev.reg_set(dev.REG_TLOW, dev.TLOW_POR)
    dev.cache_clear()


def on_board(cls):
    """A fresh ``cls`` on the real TMP102 behind mux channel 2, with the part at power-on."""
    restore(TMP102Regs(I2CConnection(bus, TMP_ADDR, TMP_PATH, bus_lock)))
    return cls(I2CConnection(bus, TMP_ADDR, TMP_PATH, bus_lock))

## The mux

With every channel off, only the mux answers: the TMP102 is hidden behind it. The cell scans
first and prints every address on the bus, so a missing mux shows what is there instead (and
says what that suggests).

**What to do:** nothing. If the mux check fails, read the hint under the scan.

In [41]:
root = bus.scan()  # scan first: if the mux is missing, show what IS on the bus
print("bus {} answers at:".format(MUX_BUS), [hex(a) for a in root])
if check("mux ACKs at {:#04x}".format(MUX_ADDR), MUX_ADDR in root):
    mux.channel_deselect()
    root = bus.scan()
    print("all channels off:", [hex(a) for a in root])
    check("TMP102 hidden behind the mux", TMP_ADDR not in root)
elif any(TCA9548A.addr_ok(a) for a in root):
    print("  a TCA9548A address answers, but not {:#04x}: set MUX_ADDR above".format(MUX_ADDR))
elif 0x29 in root or 0x18 in root:
    print("  0x29 / 0x18 answer here: the VL53L4CD/relay chain is on this bus, so the mux")
    print("  is on the other Qwiic chain: set MUX_BUS above")
else:
    print("  no mux found: run drivers_tca9548a_devNB's 'Find the mux', or check the cable")

bus 1 answers at: ['0x48', '0x70']
  [PASS] mux ACKs at 0x70                     
all channels off: ['0x70']
  [PASS] TMP102 hidden behind the mux         


With channel 2 on, the TMP102 joins the bus and the control register reads back channel 2.
If it is missing, the Qwiic cable is probably in the wrong mux port (port 0 is the TMAG5273).

**What to do:** nothing.

In [42]:
try:
    mux.channel_select(TMP_CHANNEL)
    print("channel 2 on:    ", [hex(a) for a in bus.scan()])
    check("TMP102 ACKs on channel 2", TMP_ADDR in bus.scan())
    check("control register reads back channel 2", mux.control_read() == 1 << TMP_CHANNEL)
finally:
    mux.channel_deselect()

channel 2 on:     ['0x48', '0x70']
  [PASS] TMP102 ACKs on channel 2             
  [PASS] control register reads back channel 2 


## The layer stack
The same register through each layer, innermost first. The bare `machine.I2C` read switches
the mux by hand, which is the step the connection does on every transfer.

**What to do:** nothing.

In [43]:
regs = on_board(TMP102Regs)
try:
    mux.channel_select(TMP_CHANNEL)
    raw_bus = bus.readfrom_mem(TMP_ADDR, regs.REG_TEMP, 2)
    mux.channel_deselect()
    raw_conn = regs.conn.readfrom_mem(regs.REG_TEMP, 2)
    raw_dev = regs.read_mem(regs.REG_TEMP, 2)
    print("machine.I2C  ", raw_bus, "| I2CConnection", raw_conn, "| I2CNode", raw_dev)
    check("all three layers return 2 bytes", len(raw_bus) == len(raw_conn) == len(raw_dev) == 2)
    check("mixin assembles MSB first", regs.reg_read(regs.REG_TEMP) >> 8 == raw_dev[0])
    check("temperature low nibble is padding", regs.reg_read(regs.REG_TEMP) & 0x000F == 0)
finally:
    mux.channel_deselect()
    restore(regs)

machine.I2C   b'\x19`' | I2CConnection b'\x19`' | I2CNode b'\x19`'
  [PASS] all three layers return 2 bytes      
  [PASS] mixin assembles MSB first            
  [PASS] temperature low nibble is padding    


## Matching tests: `TMP102Regs`
One cell per unit test of `TMP102Regs` above, with the same name. Tests that need no bus
traffic run too: here they check the arithmetic on MicroPython itself, which the host tests
cannot.

**Matches** `test_fields_tile_the_data_bits`: the nine config fields do not overlap and cover
bits 15..4.

**What to do:** nothing. No bus traffic.

In [44]:
covered, overlap = 0, False
for name in ("F_OS", "F_RES", "F_FQ", "F_POL", "F_TM", "F_SD", "F_CR", "F_AL", "F_EM"):
    mask = TMP102Regs.field_mask(getattr(TMP102Regs, name))
    overlap = overlap or bool(covered & mask)
    covered |= mask
check("test_fields_tile_the_data_bits", not overlap and covered == 0xFFF0, hex(covered))

  [PASS] test_fields_tile_the_data_bits       0xfff0


True

**Matches** `test_config_verify_mask_excludes_os_res_al`: the mask leaves out exactly OS,
R1/R0 and AL, and only the config register has one.

**What to do:** nothing. No bus traffic.

In [45]:
m = TMP102Regs.field_mask
excluded = m(TMP102Regs.F_OS) | m(TMP102Regs.F_RES) | m(TMP102Regs.F_AL)
check("test_config_verify_mask_excludes_os_res_al",
      0xFFFF & ~TMP102Regs.CONFIG_VERIFY_MASK == excluded
      and TMP102Regs.VERIFY_MASKS == {TMP102Regs.REG_CONFIG: TMP102Regs.CONFIG_VERIFY_MASK})  # fmt: skip

  [PASS] test_config_verify_mask_excludes_os_res_al 


True

**Matches** `test_por_values_decode_as_the_datasheet_says`, on the real part: after a reset to
power-on values the config register reads R1/R0 = `11`, a 4 Hz rate and AL = 1.

**What to do:** keep the sensor at room temperature (AL reads 0 above 80 C).

In [46]:
regs = on_board(TMP102Regs)
try:
    cfg = regs.reg_read(regs.REG_CONFIG)
    show(cfg, "config")
    x = regs.field_extract
    check("test_por_values_decode_as_the_datasheet_says",
          x(cfg, regs.F_RES) == 0b11 and regs.CONV_RATES[x(cfg, regs.F_CR)] == 4.0
          and x(cfg, regs.F_AL) == 1, "{:#06x}".format(cfg))  # fmt: skip
finally:
    restore(regs)

config           0x60a0  01100000 10100000  24736
  [PASS] test_por_values_decode_as_the_datasheet_says 0x60a0


**Matches** `test_masked_write_lands_and_exact_write_is_refused`, on the real part: R1/R0 are
read-only, so a request with them cleared fails an exact verify and lands under the class
mask.

**What to do:** nothing.

In [47]:
regs = on_board(TMP102Regs)
try:
    sabotaged = regs.CFG_POR & ~regs.field_mask(regs.F_RES)
    check_raises(
        "exact verify refused",
        ValueError,
        regs.reg_set,
        regs.REG_CONFIG,
        sabotaged,
        verify_mask=0xFFFF,
    )
    landed = regs.reg_set(regs.REG_CONFIG, sabotaged)
    check(
        "test_masked_write_lands_and_exact_write_is_refused",
        landed == regs.CFG_POR,
        "{:#06x}".format(landed),
    )
finally:
    restore(regs)

  [PASS] exact verify refused                 
          reg 0x01: wrote 0xa0, device holds 0x60a0 (mask 0xffff)
  [PASS] test_masked_write_lands_and_exact_write_is_refused 0x60a0


**Matches** `test_address_and_clock_limits`: the four ADD0 addresses are allowed, others are
not, and 1 MHz is above the part's limit.

**What to do:** nothing. No bus traffic.

In [48]:
check("test_address_and_clock_limits",
      all(TMP102Regs.addr_ok(a) for a in (0x48, 0x49, 0x4A, 0x4B)) and not TMP102Regs.addr_ok(0x70)
      and TMP102Regs.baud_ok(100_000) and not TMP102Regs.baud_ok(1_000_000))  # fmt: skip

  [PASS] test_address_and_clock_limits        


True

## Matching tests: codecs
The datasheet cases on MicroPython. The board cannot import the generated test file (it needs
pytest), so the next cell holds a copy of `CASES_12` and `CASES_13`, and the cell after it
checks the copy against the lists above.

Notebook-only: the datasheet cases, copied to the board.

**What to do:** nothing.

In [49]:
CASES_12 = [(127.9375, 0x7FF0), (100.0, 0x6400), (75.0, 0x4B00), (50.0, 0x3200),
            (25.0, 0x1900), (0.25, 0x0040), (0.0, 0x0000), (-0.25, 0xFFC0),
            (-25.0, 0xE700), (-55.0, 0xC900)]  # fmt: skip
CASES_13 = [(150.0, 0x4B00), (128.0, 0x4000), (100.0, 0x3200), (25.0, 0x0C80),
            (0.0, 0x0000), (-25.0, 0xF380), (-55.0, 0xE480)]  # fmt: skip

Notebook-only: on the host, read the board's copy back with `%eval` and check it matches the
notebook's own lists, so the two cannot drift apart.

**What to do:** nothing.

In [50]:
%%host
board_cases = get_ipython().run_line_magic("eval", "[CASES_12, CASES_13]")
host_cases = [[list(c) for c in CASES_12], [list(c) for c in CASES_13]]
if board_cases != host_cases:
    raise AssertionError("the board's copy of the datasheet cases differs from CASES_12/CASES_13")
print("board copy of the datasheet cases matches the notebook's")

board copy of the datasheet cases matches the notebook's


**Matches** `test_codec_12bit_matches_datasheet`: every 12-bit case encodes and decodes
exactly on MicroPython.

**What to do:** nothing. No bus traffic.

In [51]:
bad = [
    c
    for c, w in CASES_12
    if TMP102.celsius_to_raw(c, False) != w or TMP102.celsius_from_raw(w, False) != c
]
check(
    "test_codec_12bit_matches_datasheet",
    not bad,
    "{} cases".format(len(CASES_12)) if not bad else "bad: {}".format(bad),
)

  [PASS] test_codec_12bit_matches_datasheet   10 cases


True

**Matches** `test_codec_13bit_matches_datasheet`: the same for extended mode.

**What to do:** nothing. No bus traffic.

In [52]:
bad = [
    c
    for c, w in CASES_13
    if TMP102.celsius_to_raw(c, True) != w or TMP102.celsius_from_raw(w, True) != c
]
check(
    "test_codec_13bit_matches_datasheet",
    not bad,
    "{} cases".format(len(CASES_13)) if not bad else "bad: {}".format(bad),
)

  [PASS] test_codec_13bit_matches_datasheet   7 cases


True

**Matches** `test_codec_refuses_out_of_range`: out-of-range temperatures are refused in both
modes, by register width or by operating range.

**What to do:** nothing. No bus traffic.

In [53]:
for celsius, em in ((200.0, False), (200.0, True), (-100.0, False), (-100.0, True)):
    check_raises(
        "test_codec_refuses_out_of_range {} C em={}".format(celsius, em),
        ValueError,
        TMP102.celsius_to_raw,
        celsius,
        em,
    )

  [PASS] test_codec_refuses_out_of_range 200.0 C em=False 
          200.0 C not representable in 12-bit mode (-128.0000 to 127.9375 C)
  [PASS] test_codec_refuses_out_of_range 200.0 C em=True 
          200.0 C outside the operating range (-55.0 to 150.0 C)
  [PASS] test_codec_refuses_out_of_range -100.0 C em=False 
          -100.0 C outside the operating range (-55.0 to 150.0 C)
  [PASS] test_codec_refuses_out_of_range -100.0 C em=True 
          -100.0 C outside the operating range (-55.0 to 150.0 C)


**Matches** `test_sign_extension_needs_the_shift_first`: the decode shifts the padding out
before sign extension.

**What to do:** nothing. No bus traffic.

In [54]:
check("test_sign_extension_needs_the_shift_first",
      TMP102.celsius_from_raw(0xE700, False) == -25.0
      and TMP102.sign_extend(0xE700, 12) * TMP102.LSB_C != -25.0)  # fmt: skip

  [PASS] test_sign_extension_needs_the_shift_first 


True

## Matching tests: `TMP102`
The driver on the real part. Where the fake has an exact value (25 C) the real part has a
physical range, so these check the same behaviour with plausible bounds.

**Matches** `test_reads_and_threshold_round_trip`: a plausible reading, power-on thresholds,
thresholds set and read back, half-LSB rounding, and a reset.

**What to do:** keep your fingers off the sensor (the reading should be room temperature).

In [55]:
tmp = on_board(TMP102)
try:
    t = tmp.temperature()
    print("temperature: {:.4f} C".format(t))
    check("plausible room temperature", 5.0 <= t <= 45.0, "{:.4f} C".format(t))
    check("thresholds at power-on", (tmp.low_limit_get(), tmp.high_limit_get()) == (75.0, 80.0))
    check("thresholds round-trip", tmp.limits_set(20.0, 30.0) == (20.0, 30.0))
    landed = tmp.low_limit_set(20.03)
    check(
        "rounds to the nearest LSB", abs(landed - 20.03) <= tmp.LSB_C / 2, "{:.4f}".format(landed)
    )
    tmp.limits_reset()
    check("test_reads_and_threshold_round_trip", tmp.high_limit_get() == 80.0)
finally:
    restore(tmp)

temperature: 26.4375 C
  [PASS] plausible room temperature           26.4375 C
  [PASS] thresholds at power-on               
  [PASS] thresholds round-trip                
  [PASS] rounds to the nearest LSB            20.0000
  [PASS] test_reads_and_threshold_round_trip  


**Matches** `test_crossed_limits_are_refused`: a crossed pair is refused, and the real part's
thresholds are untouched.

**What to do:** nothing.

In [56]:
tmp = on_board(TMP102)
try:
    check_raises("test_crossed_limits_are_refused", ValueError, tmp.limits_set, 80.0, 20.0)
    check("thresholds untouched", (tmp.low_limit_get(), tmp.high_limit_get()) == (75.0, 80.0))
finally:
    restore(tmp)

  [PASS] test_crossed_limits_are_refused      
          T_LOW (80.0) must be below T_HIGH (20.0)
  [PASS] thresholds untouched                 


**Matches** `test_config_properties_round_trip`: every config property reads back from the
real part what was set.

**What to do:** nothing. The ALERT pin may toggle while this runs.

In [57]:
tmp = on_board(TMP102)
try:
    ok = True
    for hz in (0.25, 1.0, 8.0, 4.0):
        tmp.conversion_rate = hz
        ok = ok and tmp.conversion_rate == hz
    for n in (1, 2, 4, 6):
        tmp.fault_queue = n
        ok = ok and tmp.fault_queue == n
    for attr in ("thermostat_mode", "alert_polarity_high", "shutdown"):
        for state in (True, False):
            setattr(tmp, attr, state)
            ok = ok and getattr(tmp, attr) is state
    check("test_config_properties_round_trip", ok)
finally:
    restore(tmp)

  [PASS] test_config_properties_round_trip    


**Matches** `test_extended_mode_is_tracked_and_refreshable`, plus what only real silicon can
show: both decodes of the same physical temperature agree, and the part sets the EM indicator
bit (D0) in 13-bit mode.

**What to do:** keep your fingers off the sensor (both reads must see the same temperature).

In [58]:
tmp = on_board(TMP102)
try:
    t12 = tmp.temperature()
    tmp.extended_mode = True
    raw13, t13 = tmp.temperature_raw(), tmp.temperature()
    check(
        "12- and 13-bit decodes agree", abs(t13 - t12) < 0.5, "{:.4f} vs {:.4f} C".format(t12, t13)
    )
    check("EM indicator bit set", raw13 & 1 == 1, "{:#06x}".format(raw13))
    tmp.em = False  # deliberately stale
    refreshed = tmp.mode_refresh()
    tmp.extended_mode = False
    check("test_extended_mode_is_tracked_and_refreshable", refreshed is True and tmp.em is False)
finally:
    restore(tmp)

  [PASS] 12- and 13-bit decodes agree         26.6250 vs 26.6250 C
  [PASS] EM indicator bit set                 0x0d51
  [PASS] test_extended_mode_is_tracked_and_refreshable 


**Matches** `test_extended_mode_switch_waits_for_the_new_format`: the reading is the same just
before and just after switching EM, both ways (within 0.5 C: the real temperature drifts).

**What to do:** keep your fingers off the sensor. Takes about half a second.

In [59]:
tmp = on_board(TMP102)
try:
    before = tmp.temperature()
    tmp.extended_mode = True
    t13 = tmp.temperature()
    tmp.extended_mode = False
    t12 = tmp.temperature()
    check("test_extended_mode_switch_waits_for_the_new_format",
          abs(t13 - before) < 0.5 and abs(t12 - before) < 0.5,
          "{:.4f} -> {:.4f} (13-bit) -> {:.4f} C".format(before, t13, t12))  # fmt: skip
finally:
    restore(tmp)

  [PASS] test_extended_mode_switch_waits_for_the_new_format 27.0000 -> 27.0000 (13-bit) -> 27.0000 C


**Matches** `test_extended_mode_switch_in_shutdown_needs_a_one_shot`: in shutdown, after an EM
change, `temperature()` refuses until a one-shot, and the one-shot reads a plausible value.

**What to do:** keep your fingers off the sensor.

In [60]:
tmp = on_board(TMP102)
try:
    before = tmp.temperature()
    tmp.shutdown = True
    tmp.extended_mode = True
    check_raises("temperature() refused before a one-shot", RuntimeError, tmp.temperature)
    t = tmp.one_shot()
    check("test_extended_mode_switch_in_shutdown_needs_a_one_shot",
          abs(t - before) < 0.5 and abs(tmp.temperature() - before) < 0.5,
          "{:.4f} -> {:.4f} C".format(before, t))  # fmt: skip
finally:
    restore(tmp)

  [PASS] temperature() refused before a one-shot 
          extended mode changed in shutdown: the temperature register keeps the old format until the next conversion; take a one_shot() first
  [PASS] test_extended_mode_switch_in_shutdown_needs_a_one_shot 27.1250 -> 27.1250 C


**Matches** `test_one_shot_needs_shutdown_mode`: refused in continuous mode; in shutdown it
returns a plausible reading within the timeout.

**What to do:** nothing.

In [61]:
tmp = on_board(TMP102)
try:
    check_raises("one-shot refused in continuous mode", RuntimeError, tmp.one_shot)
    tmp.shutdown = True
    start = ticks_ms()
    t = tmp.one_shot()
    check("test_one_shot_needs_shutdown_mode", tmp.TEMP_MIN <= t <= tmp.TEMP_MAX,
          "{:.4f} C in {} ms".format(t, ticks_diff(ticks_ms(), start)))  # fmt: skip
finally:
    restore(tmp)

  [PASS] one-shot refused in continuous mode  
          one_shot() requires shutdown mode; set .shutdown = True
  [PASS] test_one_shot_needs_shutdown_mode    27.1250 C in 22 ms


## Hardware-only checks
`RegisterI2CBase` behaviour that the fake cannot prove: the cache, the validator, tolerance,
the raw write path, a live reading and the ALERT comparator.

The cache: a hit returns the cached value, it is width-aware, and clearing it empties it.

**What to do:** nothing.

In [62]:
regs = on_board(TMP102Regs)
try:
    first = regs.reg_get(regs.REG_TEMP)
    check("cache hit returns the cached value", regs.reg_get(regs.REG_TEMP) == first)
    check(
        "cache is width-aware",
        regs.reg_get(regs.REG_CONFIG, nbytes=1) == regs.reg_get(regs.REG_CONFIG) >> 8,
    )
    regs.cache_clear()
    check("cache cleared", not regs.reg_cache)
finally:
    restore(regs)

  [PASS] cache hit returns the cached value   
  [PASS] cache is width-aware                 
  [PASS] cache cleared                        


A field write moves only its own bits on the real register. OS is left out of the comparison:
entering shutdown finishes the conversion and OS then reads 1 by itself (a status bit, outside
the verify mask; see the diagnostic at the end).

**What to do:** nothing.

In [63]:
regs = on_board(TMP102Regs)
try:
    before = regs.reg_read(regs.REG_CONFIG)
    regs.field_set(regs.REG_CONFIG, regs.F_SD, 1)
    changed = (before ^ regs.reg_read(regs.REG_CONFIG)) & regs.CONFIG_VERIFY_MASK
    check(
        "field_set moved only SD", changed == regs.field_mask(regs.F_SD), "{:#06x}".format(changed)
    )
finally:
    restore(regs)

  [PASS] field_set moved only SD              0x0100


A validator runs before any write: a refused value leaves the real register untouched.

**What to do:** nothing.

In [64]:


def under_100c(v):
    """Validator: refuse a raw threshold word above 100 C (0x6400)."""
    if v > 0x6400:
        raise ValueError("{:#06x} exceeds 100 C".format(v))


regs = on_board(TMP102Regs)
try:
    check_raises(
        "validator fires", ValueError, regs.reg_set, regs.REG_THIGH, 0x7000, validator=under_100c
    )
    check("threshold untouched", regs.reg_read(regs.REG_THIGH) == regs.THIGH_POR)
    check(
        "accepted value verifies on every bit",
        regs.reg_set(regs.REG_THIGH, 0x1E00, validator=under_100c) == 0x1E00,
    )
finally:
    restore(regs)

  [PASS] validator fires                      
          0x7000 exceeds 100 C
  [PASS] threshold untouched                  
  [PASS] accepted value verifies on every bit 


Tolerance: R1/R0 make the config read back differently from a request with them cleared, so
the real difference is measured, then a tolerance just under it must fail and one equal to it
must pass.

**What to do:** nothing.

In [65]:
regs = on_board(TMP102Regs)
try:
    sabotaged = regs.CFG_POR & ~regs.field_mask(regs.F_RES)
    regs.reg_write(regs.REG_CONFIG, sabotaged)
    delta = abs(regs.reg_read(regs.REG_CONFIG) - sabotaged)
    check_raises(
        "tolerance just under fails",
        ValueError,
        regs.reg_set,
        regs.REG_CONFIG,
        sabotaged,
        tolerance=delta - 1,
    )
    landed = regs.reg_set(regs.REG_CONFIG, sabotaged, tolerance=delta)
    check(
        "tolerance equal to the delta passes",
        abs(landed - sabotaged) == delta,
        "delta {:#x}".format(delta),
    )
finally:
    restore(regs)

  [PASS] tolerance just under fails           
          reg 0x01: wrote 0xa0, device holds 0x60a0 (tol 24575)
  [PASS] tolerance equal to the delta passes  delta 0x6000


The raw write path bypasses the cache, which then goes stale; a refresh fixes it.

**What to do:** nothing.

In [66]:
regs = on_board(TMP102Regs)
try:
    regs.reg_get(regs.REG_THIGH, refresh=True)
    regs.reg_write(regs.REG_THIGH, 0x2800)
    check("raw write leaves the cache stale", regs.reg_get(regs.REG_THIGH) != 0x2800)
    check("refresh fixes it", regs.reg_get(regs.REG_THIGH, refresh=True) == 0x2800)
finally:
    restore(regs)

  [PASS] raw write leaves the cache stale     
  [PASS] refresh fixes it                     


A live reading tracks reality: 20 readings at 8 Hz over 5 s. The check passes if the
temperature rises by at least 0.25 C during the run.

**What to do:** run the cell, then **pinch the sensor between your fingers** right away and
hold it until the cell finishes (5 s).

In [67]:
tmp = on_board(TMP102)
try:
    tmp.conversion_rate = 8.0
    readings = []
    for _ in range(20):
        readings.append(tmp.temperature())
        print("  {:7.4f} C".format(readings[-1]))
        sleep_ms(250)
    rise = max(readings) - readings[0]
    check("reading rose while pinched", rise >= 0.25, "rise {:.4f} C".format(rise))
finally:
    restore(tmp)

  30.8750 C
  31.0625 C
  31.1875 C
  31.3125 C
  31.5000 C
  31.5625 C
  31.6875 C
  31.7500 C
  31.8750 C
  31.9375 C
  32.0000 C
  32.0000 C
  32.1250 C
  32.1250 C
  32.1875 C
  32.2500 C
  32.3125 C
  32.3125 C
  32.3125 C
  32.3750 C
  [PASS] reading rose while pinched           rise 1.5000 C


The ALERT comparator follows the thresholds: T_HIGH below ambient asserts AL, T_LOW above it
releases AL. With POL=0, AL reads **1 when not alerting**.

**What to do:** let go of the sensor and wait about 30 s for it to cool back to room
temperature before running this cell. Keep your hands off while it runs.

In [68]:
tmp = on_board(TMP102)
try:
    tmp.fault_queue = 1
    tmp.conversion_rate = 8.0
    t_now = tmp.temperature()
    check("no alert at power-on limits", tmp.alert is True)
    tmp.limits_set(t_now - 20.0, t_now - 10.0)
    sleep_ms(500)
    check("alert asserts below ambient", tmp.alert is False)
    tmp.limits_set(t_now + 10.0, t_now + 20.0)
    sleep_ms(500)
    check("alert releases above ambient", tmp.alert is True)
finally:
    restore(tmp)

  [PASS] no alert at power-on limits          
  [PASS] alert asserts below ambient          
  [PASS] alert releases above ambient         


## Diagnostics
Notebook-only: raw views of the two quirks the matching tests depend on. They print, they do
not count towards the tally.

### When does the temperature word switch to 13-bit?

Notebook-only: turns extended mode on, then reads the raw temperature word every 10 ms for about
half a second and prints each change with its time, decoded both ways. If the first words are
still 12-bit data (with the EM indicator D0 already 1) and the format only changes at the next
conversion, that is the lag the driver waits out. Measured 2026-11-13: old format at 5 ms, new at 61 ms.

**What to do:** keep your fingers off the sensor (the temperature must stay steady).

In [69]:
regs = on_board(TMP102Regs)  # register level: the driver would wait out the lag
try:
    print("conversion rate: 4 Hz (power-on)")
    t0 = ticks_ms()
    regs.field_set(regs.REG_CONFIG, regs.F_EM, 1)
    seen = []
    for _ in range(50):  # about 500 ms, two conversions at 4 Hz
        seen.append((ticks_diff(ticks_ms(), t0), regs.reg_read(regs.REG_TEMP)))
        sleep_ms(10)
    prev = None
    for ms, raw in seen:
        if raw != prev:  # print only when the word changes
            print(
                "{:4d} ms  {:#06x}  D0={}  as 12-bit {:8.4f} C  as 13-bit {:8.4f} C".format(
                    ms,
                    raw,
                    raw & 1,
                    TMP102.celsius_from_raw(raw, False),
                    TMP102.celsius_from_raw(raw, True),
                )
            )
            prev = raw
finally:
    restore(regs)

conversion rate: 4 Hz (power-on)
   5 ms  0x1dd1  D0=1  as 12-bit  29.8125 C  as 13-bit  59.6250 C
 195 ms  0x0ee9  D0=1  as 12-bit  14.8750 C  as 13-bit  29.8125 C


### The OS bit when entering shutdown

Notebook-only: shows the config register before and after setting SD. If OS goes from 0 to 1 by
itself after shutdown, `field_set` is fine and the "moved only SD" check must ignore OS (a status
bit, already outside the verify mask).

**What to do:** nothing.

In [70]:
regs = on_board(TMP102Regs)
try:
    show(regs.reg_read(regs.REG_CONFIG), "continuous")
    regs.field_set(regs.REG_CONFIG, regs.F_SD, 1)
    for i in range(4):
        cfg = regs.reg_read(regs.REG_CONFIG)
        show(cfg, "shutdown +{} ms".format(i * 20))
        print("               OS = {}".format(regs.field_extract(cfg, regs.F_OS)))
        sleep_ms(20)
finally:
    restore(regs)

continuous       0x60a0  01100000 10100000  24736
shutdown +0 ms   0x61a0  01100001 10100000  24992
               OS = 0
shutdown +20 ms  0xe1a0  11100001 10100000  57760
               OS = 1
shutdown +40 ms  0xe1a0  11100001 10100000  57760
               OS = 1
shutdown +60 ms  0xe1a0  11100001 10100000  57760
               OS = 1


## Tally and disconnect

Notebook-only: the PASS/FAIL tally of every check in this section.

**What to do:** read it. Any `FAIL` line above says which check and why.

In [71]:
check_summary()

51 passed, 0 failed


(51, 0)

Notebook-only: copy the tally to the host before disconnecting, so the last cell can fail
the run after the board is released.

**What to do:** nothing.

In [72]:
%%host
hw_passed, hw_failed = get_ipython().run_line_magic("eval", "check_counts()")

Notebook-only: release the board, so the next run (or a kernel restart) starts on the host.

**What to do:** nothing.

In [73]:
%board disconnect

Disconnected /dev/ttyACM0 (port closed, board not reset)


Notebook-only: fail the run if any hardware check failed. It runs on the host, so it works
whether or not the board is still attached.

**What to do:** nothing. An error here means a `FAIL` line above.

In [74]:
%%host
if hw_failed:
    raise AssertionError("{} of {} hardware checks failed".format(hw_failed, hw_passed + hw_failed))
print("all {} hardware checks passed".format(hw_passed))

all 51 hardware checks passed
